# T5 Question Generation — Fine-Tuning on SQuAD

**Project:** VR-Enhanced Interview Preparation System  
**Model:** `t5-small` fine-tuned for answer-aware Question Generation  
**Dataset:** SQuAD v1.1 (Stanford Question Answering Dataset)  

**Pipeline:**
```
Resume PDF  →  DeBERTa NER  →  Context Builder  →  T5-QG (this model)  →  Interview Question
```

> ⚠️ Set runtime to **GPU (T4)** before running: Runtime → Change runtime type → T4 GPU


## Cell 1 — Install Dependencies (pinned versions, no conflicts)

In [ ]:
# Uninstall any pre-installed conflicting versions first
import subprocess, sys

packages_to_remove = ['transformers', 'datasets', 'evaluate', 'accelerate']
for pkg in packages_to_remove:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg],
                   capture_output=True)

print('Removed old versions. Installing pinned versions...')


In [ ]:
# Install pinned, mutually compatible versions
!pip install -q \
    transformers==4.57.4 \
    datasets==3.6.0 \
    evaluate==0.4.3 \
    accelerate==1.13.0 \
    rouge_score \
    sentencepiece \
    sacrebleu

print('Done! Now RESTART the runtime: Runtime → Restart session')
print('Then run from Cell 2 onwards (skip Cell 1).')


> **After Cell 1 completes → Runtime → Restart session → then run Cell 2 onwards.**  
> Do NOT re-run Cell 1 after restarting.


## Cell 2 — Imports & GPU Check

In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, HTML

import torch

import transformers
print(f'transformers version : {transformers.__version__}')
assert transformers.__version__.startswith('4.57'), \
    f'Wrong version {transformers.__version__}. Re-run Cell 1 and restart runtime.'

from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,              # replaces deprecated T5Tokenizer
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from datasets import load_dataset
import evaluate

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU  : {gpu_name}')
    print(f'VRAM : {gpu_mem:.1f} GB')
else:
    print('WARNING: No GPU — switch to T4 GPU runtime!')

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {device}')
print('All imports OK.')


## Cell 3 — Configuration

In [ ]:
CFG = {
    # Model
    'model_name'      : 't5-small',
    'max_input_len'   : 512,
    'max_target_len'  : 64,

    # Dataset
    'dataset_name'    : 'squad',
    'train_samples'   : 8000,
    'val_samples'     : 1000,
    'seed'            : 42,

    # Training
    'epochs'          : 5,
    'batch_size'      : 8,
    'grad_accum'      : 4,      # effective batch = 32
    'lr'              : 3e-4,
    'weight_decay'    : 0.01,
    'warmup_ratio'    : 0.06,
    'fp16'            : True,

    # Output
    'output_dir'      : './t5-qg-squad',
    'logging_steps'   : 50,
}

os.makedirs(CFG['output_dir'], exist_ok=True)

print('Configuration:')
for k, v in CFG.items():
    print(f'  {k:<22}: {v}')


## Cell 4 — Load SQuAD Dataset

In [ ]:
print('Downloading SQuAD v1.1 ...')
raw = load_dataset('squad')

print(f'Train      : {len(raw["train"]):,} examples')
print(f'Validation : {len(raw["validation"]):,} examples')

ex = raw['train'][0]
print('\nSample example:')
print(f'  Context  : {ex["context"][:120]}...')
print(f'  Question : {ex["question"]}')
print(f'  Answer   : {ex["answers"]["text"][0]}')


## Cell 5 — Tokenizer & Preprocessing

In [ ]:
print(f'Loading tokenizer: {CFG["model_name"]}')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

def preprocess(batch):
    """
    Source: 'generate question: answer: <ANS> context: <CTX>'
    Target: '<QUESTION>'
    """
    inputs, targets = [], []
    for ctx, q, ans_dict in zip(batch['context'], batch['question'], batch['answers']):
        answer = ans_dict['text'][0] if ans_dict['text'] else ''
        inputs.append(f'generate question: answer: {answer} context: {ctx}')
        targets.append(q)

    model_inputs = tokenizer(
        inputs,
        max_length=CFG['max_input_len'],
        padding='max_length',
        truncation=True,
    )
    labels = tokenizer(
        targets,
        max_length=CFG['max_target_len'],
        padding='max_length',
        truncation=True,
    )
    # Replace padding id with -100 so loss ignores pad tokens
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in lbl]
        for lbl in labels['input_ids']
    ]
    return model_inputs

print(f'Tokenizing {CFG["train_samples"]:,} train + {CFG["val_samples"]:,} val examples ...')
t0 = time.time()

train_small = raw['train'].shuffle(seed=CFG['seed']).select(range(CFG['train_samples']))
val_small   = raw['validation'].shuffle(seed=CFG['seed']).select(range(CFG['val_samples']))

tok_train = train_small.map(
    preprocess, batched=True, batch_size=256,
    remove_columns=train_small.column_names
)
tok_val = val_small.map(
    preprocess, batched=True, batch_size=256,
    remove_columns=val_small.column_names
)

tok_train.set_format('torch')
tok_val.set_format('torch')

print(f'Done in {time.time()-t0:.1f}s')
print(f'input_ids shape : {tok_train[0]["input_ids"].shape}')
print(f'labels shape    : {tok_train[0]["labels"].shape}')


## Cell 6 — Load Model

In [ ]:
print(f'Loading model: {CFG["model_name"]}')
model = T5ForConditionalGeneration.from_pretrained(CFG['model_name'])
model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total:,}')
print(f'Trainable parameters : {trainable:,}')
print(f'Approx size (fp32)   : {total * 4 / 1e6:.1f} MB')


## Cell 7 — Metrics (ROUGE)

In [ ]:
rouge = evaluate.load('rouge')

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # ── FIX: clamp prediction IDs to valid vocab range ──────────────────
    # generate() can produce IDs >= vocab_size (e.g. 32100 for t5-small
    # which has exactly 32100 tokens). SentencePiece raises IndexError
    # on those. Clamp to [0, vocab_size - 1] before decoding.
    vocab_size = tokenizer.vocab_size          # 32100 for t5-small
    preds = np.clip(preds, 0, vocab_size - 1)

    # Replace -100 padding back to pad_token_id for decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, vocab_size - 1)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    result = {k: round(v * 100, 2) for k, v in result.items()}
    result['gen_len'] = round(np.mean([np.count_nonzero(p) for p in preds]), 2)
    return result

print('Metrics ready: ROUGE-1, ROUGE-2, ROUGE-L, ROUGE-Lsum, gen_len')


## Cell 8 — Training Arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir                  = CFG['output_dir'],
    num_train_epochs            = CFG['epochs'],
    per_device_train_batch_size = CFG['batch_size'],
    per_device_eval_batch_size  = CFG['batch_size'] * 2,
    gradient_accumulation_steps = CFG['grad_accum'],
    learning_rate               = CFG['lr'],
    weight_decay                = CFG['weight_decay'],
    warmup_ratio                = CFG['warmup_ratio'],
    fp16                        = CFG['fp16'] and (device == 'cuda'),
    predict_with_generate       = True,
    generation_max_length       = CFG['max_target_len'],
    generation_num_beams        = 4,       # use beam search during eval decode
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    logging_dir                 = f'{CFG["output_dir"]}/logs',
    logging_steps               = CFG['logging_steps'],
    load_best_model_at_end      = True,
    metric_for_best_model       = 'rouge1',
    greater_is_better           = True,
    save_total_limit            = 2,
    report_to                   = 'none',
    seed                        = CFG['seed'],
    dataloader_num_workers      = 2,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

eff_batch   = CFG['batch_size'] * CFG['grad_accum']
steps_ep    = CFG['train_samples'] // eff_batch
total_steps = steps_ep * CFG['epochs']
print(f'Effective batch size : {eff_batch}')
print(f'Steps per epoch      : {steps_ep}')
print(f'Total steps          : {total_steps}')
print(f'Warmup steps         : {int(total_steps * CFG["warmup_ratio"])}')


## Cell 9 — Train the Model

In [ ]:
trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tok_train,
    eval_dataset    = tok_val,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training ...')
print('=' * 55)
t_start = time.time()

train_result = trainer.train()

duration = time.time() - t_start
print('=' * 55)
print(f'Training complete in {duration/60:.1f} minutes')
print(f'Final train loss : {train_result.training_loss:.4f}')
print(f'Total steps      : {train_result.global_step}')


## Cell 10 — Training Curves (screenshot this for your slides)

In [ ]:
log_history = trainer.state.log_history

train_rows = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_rows  = [l for l in log_history if 'eval_loss' in l]

train_steps  = [r['step'] for r in train_rows]
train_losses = [r['loss'] for r in train_rows]
eval_epochs  = [r['epoch'] for r in eval_rows]
eval_losses  = [r['eval_loss'] for r in eval_rows]
eval_rouge1  = [r.get('eval_rouge1', 0) for r in eval_rows]
eval_rougeL  = [r.get('eval_rougeL', 0) for r in eval_rows]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('T5-Small Fine-Tuning on SQuAD  |  VR Interview System',
             fontsize=14, fontweight='bold')

# --- Plot 1: step-level train loss ---
ax = axes[0]
ax.plot(train_steps, train_losses, color='#2563EB', linewidth=2)
ax.fill_between(train_steps, train_losses, alpha=0.1, color='#2563EB')
ax.set_title('Training Loss (per 50 steps)', fontweight='bold')
ax.set_xlabel('Step'); ax.set_ylabel('Cross-Entropy Loss')
ax.grid(True, linestyle='--', alpha=0.4)
if train_losses:
    ax.annotate(f'Start\n{train_losses[0]:.3f}',
                xy=(train_steps[0], train_losses[0]),
                xytext=(train_steps[0]+5, train_losses[0]+0.08),
                fontsize=8, color='#2563EB',
                arrowprops=dict(arrowstyle='->', color='#2563EB'))
    ax.annotate(f'End\n{train_losses[-1]:.3f}',
                xy=(train_steps[-1], train_losses[-1]),
                xytext=(train_steps[-1]-80, train_losses[-1]+0.08),
                fontsize=8, color='green',
                arrowprops=dict(arrowstyle='->', color='green'))

# --- Plot 2: train vs eval loss per epoch ---
n_epochs = len(eval_epochs)
ep_size  = max(len(train_steps) // n_epochs, 1) if n_epochs else 1
train_per_epoch = [
    train_losses[min(i * ep_size, len(train_losses)-1)]
    for i in range(1, n_epochs + 1)
]
ax2 = axes[1]
ax2.plot(eval_epochs, train_per_epoch, 'o-', color='#2563EB',
         linewidth=2, markersize=6, label='Train Loss')
ax2.plot(eval_epochs, eval_losses,     's--', color='#DC2626',
         linewidth=2, markersize=6, label='Eval Loss')
ax2.set_title('Train vs Eval Loss (per Epoch)', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_xticks(range(1, n_epochs + 1))
ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.4)

# --- Plot 3: ROUGE scores ---
ax3 = axes[2]
ax3.plot(eval_epochs, eval_rouge1, 'D-', color='#16A34A',
         linewidth=2, markersize=6, label='ROUGE-1')
ax3.plot(eval_epochs, eval_rougeL, '^-', color='#9333EA',
         linewidth=2, markersize=6, label='ROUGE-L')
ax3.set_title('ROUGE Scores (per Epoch)', fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('ROUGE Score')
ax3.set_xticks(range(1, n_epochs + 1))
ax3.legend(); ax3.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plot_path = f'{CFG["output_dir"]}/training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')
print('SCREENSHOT THIS — training evidence for your evaluators!')


## Cell 11 — Metrics Table (screenshot this too)

In [ ]:
rows = []
for er in eval_rows:
    rows.append({
        'Epoch'      : int(er.get('epoch', 0)),
        'Eval Loss'  : round(er.get('eval_loss', 0), 4),
        'ROUGE-1'    : round(er.get('eval_rouge1',  0), 2),
        'ROUGE-2'    : round(er.get('eval_rouge2',  0), 2),
        'ROUGE-L'    : round(er.get('eval_rougeL',  0), 2),
        'Gen Len'    : round(er.get('eval_gen_len',  0), 1),
    })

df = pd.DataFrame(rows)
best_idx = df['ROUGE-1'].idxmax() if not df.empty else 0

def highlight_best(row):
    return ['background-color:#d1fae5;font-weight:bold'] * len(row) \
           if row.name == best_idx else [''] * len(row)

print('Per-Epoch Evaluation Metrics  (green = best epoch):')
display(df.style.apply(highlight_best, axis=1)
          .set_caption('T5-Small | SQuAD QG | Green = Best Checkpoint'))

if not df.empty:
    print(f'\nBest epoch : {df.loc[best_idx, "Epoch"]}')
    print(f'ROUGE-1    : {df.loc[best_idx, "ROUGE-1"]}')
    print(f'ROUGE-2    : {df.loc[best_idx, "ROUGE-2"]}')
    print(f'ROUGE-L    : {df.loc[best_idx, "ROUGE-L"]}')
    print(f'Eval Loss  : {df.loc[best_idx, "Eval Loss"]}')
print('SCREENSHOT THIS TABLE for your presentation!')


## Cell 12 — Save Model & Tokenizer

In [ ]:
final_path = f'{CFG["output_dir"]}/final_model'
os.makedirs(final_path, exist_ok=True)

print(f'Saving best model to: {final_path}')
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

with open(f'{final_path}/training_config.json', 'w') as f:
    json.dump(CFG, f, indent=2)

final_metrics = trainer.evaluate()
with open(f'{final_path}/eval_results.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print('Saved files:')
for fname in sorted(os.listdir(final_path)):
    sz = os.path.getsize(f'{final_path}/{fname}')
    print(f'  {fname:<35}  {sz/1e6:6.2f} MB')


## Cell 13 — Generate Sample Questions (live inference)

In [ ]:
from transformers import pipeline as hf_pipeline

qg = hf_pipeline(
    'text2text-generation',
    model=final_path,
    tokenizer=final_path,
    device=0 if device == 'cuda' else -1,
)

def generate_question(skill, context=None):
    if context is None:
        context = (f'The candidate has experience with {skill} '
                   f'and has used it in production web development projects.')
    prompt = f'generate question: answer: {skill} context: {context}'
    out = qg(prompt, max_new_tokens=64, num_beams=4, early_stopping=True)
    return out[0]['generated_text']

test_cases = [
    ('React',       'Candidate built a React SPA with Redux and REST API integration.'),
    ('Python',      'Candidate used Python for backend development with FastAPI.'),
    ('REST APIs',   'Candidate designed RESTful microservices deployed on AWS Lambda.'),
    ('Docker',      'Candidate containerised applications using Docker Compose.'),
    ('PostgreSQL',  'Candidate optimised PostgreSQL queries and managed migrations.'),
]

print('Generated Interview Questions from Resume Skills:')
print('-' * 65)
results = []
for skill, ctx in test_cases:
    q = generate_question(skill, ctx)
    results.append({'Skill (DeBERTa NER output)': skill, 'Generated Question': q})
    print(f'Skill   : {skill}')
    print(f'Context : {ctx}')
    print(f'Q       : {q}')
    print('-' * 65)

display(pd.DataFrame(results))


## Cell 14 — Download Model (zip)

In [ ]:
import shutil
from google.colab import files

zip_base = './t5_qg_squad_model'
print('Zipping model folder ...')
shutil.make_archive(zip_base, 'zip', final_path)

zip_size = os.path.getsize(zip_base + '.zip') / 1e6
print(f'Zip size : {zip_size:.1f} MB')
print('Downloading ...')
files.download(zip_base + '.zip')
print('Download triggered!')
print()
print('Unzip to get:')
print('  config.json              model architecture')
print('  pytorch_model.bin        trained weights')
print('  tokenizer.json')
print('  spiece.model')
print('  training_config.json     hyperparameters')
print('  eval_results.json        final metrics')


## Cell 15 — FastAPI endpoint (paste into your backend)

In [ ]:
code = '''
# backend/routers/question_gen.py
# Place unzipped model folder at: backend/models/t5-qg/

from fastapi import APIRouter, HTTPException
from pydantic import BaseModel
from transformers import pipeline
import torch

router = APIRouter(prefix="/api", tags=["Question Generation"])
MODEL_PATH = "./models/t5-qg"
_pipe = None

def get_pipe():
    global _pipe
    if _pipe is None:
        _pipe = pipeline(
            "text2text-generation",
            model=MODEL_PATH,
            tokenizer=MODEL_PATH,
            device=0 if torch.cuda.is_available() else -1,
        )
    return _pipe

class QGRequest(BaseModel):
    skill: str
    context: str

class QGResponse(BaseModel):
    question: str
    skill: str

@router.post("/generate-question", response_model=QGResponse)
async def generate_question(req: QGRequest):
    try:
        prompt = f"generate question: answer: {req.skill} context: {req.context}"
        out = get_pipe()(prompt, max_new_tokens=64, num_beams=4, early_stopping=True)
        return QGResponse(question=out[0]["generated_text"], skill=req.skill)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''
print(code)


---
## Evaluator checklist

| # | Item | Where |
|---|------|-------|
| 1 | Training loss curve screenshot | Cell 10 |
| 2 | Epoch metrics table screenshot | Cell 11 |
| 3 | Model files downloaded | Cell 14 |
| 4 | FastAPI endpoint | Cell 15 |

### Dataset used
- **Name:** SQuAD v1.1 (Stanford Question Answering Dataset)
- **Source:** `load_dataset('squad')` from Hugging Face Hub
- **Task format:** Answer-aware QG — `generate question: answer: X context: Y`
- **Split used:** 8,000 train / 1,000 validation

### Expected results on T4 GPU
| Metric | Expected range |
|--------|---------------|
| Final train loss | 1.0 – 1.4 |
| Final eval loss  | 1.8 – 2.4 |
| ROUGE-1          | 18 – 28    |
| ROUGE-L          | 16 – 25    |
| Training time    | 30 – 45 min |
